# Visual Ingredient Scanner — YOLO11s Training on Google Colab

**Course:** Computer Vision · Master MEI FIB · UPC · Spring 2026  
**Team:** Pol Plana, Emma Nájera, Houda El Fezzak

This notebook:
1. Downloads all 4 Roboflow datasets
2. Merges them into one dataset, applying all class renames automatically
3. Trains YOLO11s for 50 epochs
4. Evaluates mAP and plots training curves
5. Exports to TFLite INT8 + CoreML
6. Saves everything to Google Drive

### Before running
1. Set runtime to **T4 GPU**: `Runtime → Change runtime type → T4 GPU`
2. Add your Roboflow API key as a Colab secret (🔑 icon → `ROBOFLOW_API_KEY`)
3. Fill in the workspace slugs in **Cell 4** (the `YOUR_WORKSPACE` placeholders)
4. Run all cells top to bottom

## 1 · Install dependencies

In [ ]:
!pip install -q ultralytics roboflow pyyaml
print("Done.")

## 2 · Verify GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — change runtime to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/visual-ingredient-scanner"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive folder ready: {DRIVE_DIR}")

## 4 · Dataset configuration

Fill in the `YOUR_WORKSPACE` placeholders with the workspace slugs from the Roboflow URLs.  
The workspace slug is the part after `universe.roboflow.com/` in the dataset URL.

Also check the **version numbers** — open each dataset on Roboflow and look at the version tab.

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

# ── Master class list — 68 classes (matches data/classes.yaml in repo) ──────
MASTER_CLASSES = [
    # fruits
    "apple","avocado","banana","blackberries","blueberries","cantaloupe",
    "coconut","fig","grapes","grapefruit","kiwi","lemon","lime","mango",
    "orange","peach","pear","pineapple","pomegranate","raspberries",
    "strawberries","watermelon",
    # vegetables
    "artichoke","beet","broccoli","brussels_sprouts","cabbage","carrot",
    "cauliflower","celery","chili","corn","cucumber","eggplant","garlic",
    "ginger","green_beans","lettuce","mushrooms","okra","onion","peas",
    "pepper","potato","pumpkin","radish","spinach","sweet_potato","tomato",
    "zucchini",
    # proteins & dairy
    "beef","butter","cheese","chicken","egg","fish","ham","heavy_cream",
    "pork","shrimp","tofu","yogurt",
    # pantry & packaged
    "bread","cereal","chocolate","coffee","flour","honey","hummus","jam",
    "juice","mayonnaise","milk","nuts","oil","pasta","rice","soda",
    "sugar","tea","tomato_sauce","vinegar","water",
]
MASTER_IDX = {name: i for i, name in enumerate(MASTER_CLASSES)}

# ── Per-dataset download + rename config ─────────────────────────────────────
# rename: applied AFTER lowercasing the source class name
# skip:   set of lowercased class names to discard entirely
DATASET_CONFIGS = [
    {
        "workspace": "yasxhed",
        "project":   "ingredient-detection-unorginazed-data",
        "version":   1,
        "rename": {
            "mayonaise":         "mayonnaise",
            "humus":             "hummus",
            "green beans":       "green_beans",
            "goat_cheese":       "cheese",
            "mozzarella cheese": "cheese",
            "olive_oil":         "oil",
        },
        "skip": {"bittergourd", "chayote", "meat"},
    },
    {
        "workspace": "YOUR_WORKSPACE",   # ← fill in (from URL: universe.roboflow.com/WORKSPACE/...)
        "project":   "veggies-and-fruits-balanced-0g1ss",
        "version":   1,
        "rename": {
            "bell pepper": "pepper",
            "salad":       "lettuce",
            "common fig":  "fig",
        },
        "skip": {"winter melon"},
    },
    {
        "workspace": "YOUR_WORKSPACE",   # ← fill in
        "project":   "vegetables-g9p5a",
        "version":   1,
        "rename": {
            "vegetable marrow": "zucchini",
            "brus capusta":     "brussels_sprouts",
            "cayliflower":      "cauliflower",
            "rediska":          "radish",
            "redka":            "radish",
            "fasol":            "beans",
            "chilli":           "chili",
            "hot pepper":       "chili",
            "salad":            "lettuce",
            "bell pepper":      "pepper",
        },
        "skip": {"banana_pacche", "squash-patisson", "burger"},
    },
    {
        "workspace": "sebastian-krauss",
        "project":   "groceries-mts9o",
        "version":   1,
        "rename": {},
        "skip": {"cake", "candy", "chips", "spices"},
    },
]

print(f"Master class list: {len(MASTER_CLASSES)} classes")
print(f"Datasets to merge: {len(DATASET_CONFIGS)}")

## 5 · Download all datasets from Roboflow

In [ ]:
from roboflow import Roboflow
from pathlib import Path
import yaml

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
downloaded = []  # list of (dataset_location, config)

for i, cfg in enumerate(DATASET_CONFIGS):
    print(f"\nDownloading [{i+1}/{len(DATASET_CONFIGS)}]: {cfg['project']} ...")
    location = f"/content/datasets/ds{i}_{cfg['project']}"
    project = rf.workspace(cfg["workspace"]).project(cfg["project"])
    dataset = project.version(cfg["version"]).download("yolov11", location=location)
    downloaded.append((dataset.location, cfg))
    print(f"  → saved to {dataset.location}")

print("\nAll datasets downloaded.")

## 6 · Merge datasets with class remapping

For each dataset:
- Lowercases all class names
- Applies the rename map
- Skips excluded classes
- Remaps class indices in every label file to match the master list
- Copies images + remapped labels into a unified directory

In [ ]:
import shutil

MERGED_DIR = Path("/content/datasets/merged")

def build_remap(source_names, rename_map, skip_set):
    """Return dict: source_idx -> master_idx, or -1 to discard."""
    remap = {}
    for i, name in enumerate(source_names):
        norm = name.lower().strip()
        if norm in skip_set:
            remap[i] = -1
            continue
        norm = rename_map.get(norm, norm)
        remap[i] = MASTER_IDX.get(norm, -1)  # -1 if not in master list
    return remap

def remap_label(src, dst, remap):
    """Rewrite a YOLO .txt label file with remapped class indices."""
    lines_out = []
    for line in Path(src).read_text().strip().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        master_idx = remap.get(int(parts[0]), -1)
        if master_idx >= 0:
            lines_out.append(f"{master_idx} " + " ".join(parts[1:]))
    Path(dst).write_text("\n".join(lines_out))

def merge_split(src_dir, split, remap, prefix):
    img_src = Path(src_dir) / split / "images"
    lbl_src = Path(src_dir) / split / "labels"
    img_dst = MERGED_DIR / split / "images"
    lbl_dst = MERGED_DIR / split / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    if not img_src.exists():
        return 0

    count = 0
    for img_path in img_src.iterdir():
        # prefix filename to avoid collisions between datasets
        new_stem = f"ds{prefix}_{img_path.stem}"
        shutil.copy(img_path, img_dst / (new_stem + img_path.suffix))
        lbl_path = lbl_src / (img_path.stem + ".txt")
        if lbl_path.exists():
            remap_label(lbl_path, lbl_dst / (new_stem + ".txt"), remap)
        count += 1
    return count

# ── Run merge ────────────────────────────────────────────────────────────────
total = {"train": 0, "valid": 0, "test": 0}

for ds_idx, (location, cfg) in enumerate(downloaded):
    data_yaml = yaml.safe_load(Path(location, "data.yaml").read_text())
    source_names = data_yaml["names"]
    remap = build_remap(source_names, cfg["rename"], cfg["skip"])

    kept   = sum(1 for v in remap.values() if v >= 0)
    skipped = sum(1 for v in remap.values() if v < 0)
    print(f"\nDataset {ds_idx+1} ({cfg['project']}):")
    print(f"  source classes: {len(source_names)}  kept: {kept}  skipped: {skipped}")

    for split in ("train", "valid", "test"):
        n = merge_split(location, split, remap, ds_idx)
        total[split] += n

# Write unified data.yaml
merged_yaml = {
    "path":  str(MERGED_DIR),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(MASTER_CLASSES),
    "names": MASTER_CLASSES,
}
MERGED_YAML = str(MERGED_DIR / "data.yaml")
with open(MERGED_YAML, "w") as f:
    yaml.dump(merged_yaml, f, default_flow_style=False, sort_keys=False)

print(f"\n{'='*50}")
print(f"Merged dataset — {MERGED_DIR}")
print(f"  train : {total['train']} images")
print(f"  valid : {total['valid']} images")
print(f"  test  : {total['test']} images")
print(f"  total : {sum(total.values())} images")
print(f"  classes: {len(MASTER_CLASSES)}")

## 7 · Verify merged dataset

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Count instances per class in training set
counter = Counter()
label_dir = MERGED_DIR / "train" / "labels"
for lf in label_dir.glob("*.txt"):
    for line in lf.read_text().strip().splitlines():
        if line:
            counter[int(line.split()[0])] += 1

labels  = [MASTER_CLASSES[i] for i in sorted(counter)]
counts  = [counter[i] for i in sorted(counter)]
missing = [c for c in MASTER_CLASSES if MASTER_IDX[c] not in counter]

print(f"Classes with data: {len(counter)} / {len(MASTER_CLASSES)}")
if missing:
    print(f"Classes with NO training data: {missing}")

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(labels, counts, color="steelblue")
ax.axhline(100, color="red", linestyle="--", label="min threshold (100)")
ax.set_title("Training instances per class — merged dataset")
ax.set_ylabel("Instances")
ax.legend()
plt.xticks(rotation=60, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 8 · Train YOLO11s

Expected time on T4: **~1–2 hours** depending on total image count.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # downloads base checkpoint automatically

results = model.train(
    data=MERGED_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    cos_lr=True,
    augment=True,
    project="runs/train",
    name="food_detector",
    exist_ok=True,
)

BEST_PT = "runs/train/food_detector/weights/best.pt"
print(f"\nBest checkpoint: {BEST_PT}")

## 9 · Evaluate — preliminary metrics

Target: **mAP50-95 > 0.40**

In [ ]:
trained = YOLO(BEST_PT)
metrics = trained.val(data=MERGED_YAML, split="test", device=0)

print(f"\nmAP50-95 : {metrics.box.map:.3f}  (target > 0.40)")
print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP75    : {metrics.box.map75:.3f}")
print()
print("Per-class mAP50-95:")
for name, ap in zip(metrics.names.values(), metrics.box.maps):
    flag = "✅" if ap >= 0.40 else "⚠️ "
    print(f"  {flag} {name:<22} {ap:.3f}")

## 10 · Plot training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("runs/train/food_detector/results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df["epoch"], df["train/box_loss"], label="train")
axes[0].plot(df["epoch"], df["val/box_loss"],   label="val")
axes[0].set_title("Box loss"); axes[0].legend()

axes[1].plot(df["epoch"], df["train/cls_loss"], label="train")
axes[1].plot(df["epoch"], df["val/cls_loss"],   label="val")
axes[1].set_title("Class loss"); axes[1].legend()

axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[2].plot(df["epoch"], df["metrics/mAP50(B)"],    label="mAP50")
axes[2].axhline(0.40, color="red", linestyle="--", label="target 0.40")
axes[2].set_title("mAP"); axes[2].legend()

plt.suptitle("Training curves — YOLO11s food detector")
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/training_curves.png", dpi=150)
plt.show()
print("Saved training_curves.png to Drive")

## 11 · Export to TFLite (Android) and CoreML (iOS)

In [ ]:
trained.export(format="tflite", int8=True, imgsz=640, data=MERGED_YAML)
print("TFLite INT8 export done.")

In [ ]:
try:
    trained.export(format="coreml", imgsz=640)
    print("CoreML export done.")
except Exception as e:
    print(f"CoreML skipped (needs macOS): {e}")

## 12 · Save all outputs to Google Drive

In [ ]:
import shutil
from pathlib import Path

weights_dir = Path("runs/train/food_detector/weights")

files = [
    (weights_dir / "best.pt",   "food_detector.pt"),
    (weights_dir / "last.pt",   "food_detector_last.pt"),
    (Path("runs/train/food_detector/results.csv"), "results.csv"),
]
for tflite in weights_dir.glob("*.tflite"):
    files.append((tflite, "food_detector_int8.tflite"))

for src, dst_name in files:
    if src.exists():
        shutil.copy(src, f"{DRIVE_DIR}/{dst_name}")
        print(f"Saved: {dst_name}")
    else:
        print(f"Not found (skipped): {src.name}")

# Also save the merged data.yaml so you can reproduce evaluation later
shutil.copy(MERGED_YAML, f"{DRIVE_DIR}/merged_data.yaml")
print("Saved: merged_data.yaml")
print("\nAll outputs saved to Google Drive ✅")

## 13 · Quick inference test

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

val_img_dir = MERGED_DIR / "valid" / "images"
test_imgs = random.sample(sorted(val_img_dir.glob("*")), min(4, len(list(val_img_dir.glob("*")))))

fig, axes = plt.subplots(1, len(test_imgs), figsize=(16, 5))
if len(test_imgs) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_imgs):
    preds = trained(str(img_path), conf=0.25, verbose=False)[0]
    img = PILImage.open(img_path)
    ax.imshow(img)
    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls  = preds.names[int(box.cls)]
        conf = float(box.conf)
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{cls} {conf:.2f}", color="lime", fontsize=8)
    ax.axis("off")

plt.suptitle("Inference on validation images", fontsize=14)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/inference_samples.png", dpi=150)
plt.show()
print("Saved inference_samples.png to Drive")

## Summary

After running this notebook you should have in `MyDrive/visual-ingredient-scanner/`:

| File | Description |
|---|---|
| `food_detector.pt` | Fine-tuned YOLO11s checkpoint |
| `food_detector_last.pt` | Last epoch checkpoint (backup) |
| `food_detector_int8.tflite` | INT8 TFLite export for Android |
| `results.csv` | Per-epoch training metrics |
| `training_curves.png` | Loss + mAP plots |
| `inference_samples.png` | Sample detection results |
| `merged_data.yaml` | Merged dataset config (for re-running eval) |

### Next steps
1. Download `food_detector.pt` → place at `models/yolo/food_detector.pt` in the repo
2. Download `food_detector_int8.tflite` → place at `models/yolo/food_detector.tflite`
3. Create `.env` with `GEMINI_API_KEY=your_key`
4. Run the Gradio demo: `python prototype/app.py`
5. Add the mAP number to `docs/phase2_report.md` Section 6